In [81]:
#pip install  climetlab

In [82]:
#pip install xskillscore

In [83]:
#pip install --upgrade fair-research-login 

In [84]:
#pip install geocat.viz

In [85]:
#conda install -c conda-forge xcdat

In [86]:
#pip install numpy

In [87]:
#pip install tropycal

In [88]:
#pip install climetlab

In [89]:
# === Core Python ===
import os
import glob
import collections
from datetime import datetime

# === Numerical & Data Handling ===
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xcdat as xcd
import xskillscore as xs

# === Scientific Computation ===
from scipy.stats import pearsonr
from scipy.signal import butter, filtfilt, sosfilt, lfilter
from scipy.optimize import curve_fit
from skimage.feature import peak_local_max
import metpy.calc as mpcalc  # Meteorology/thermodynamics

# === Plotting & Visualization ===
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
from matplotlib.patches import Polygon
import matplotlib.colors as mcolors
import matplotlib.path as mpath
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec

# === Mapping & Cartography ===
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# === Visualization Helpers ===
import geocat.viz.util as gvutil
import geocat.viz as gv
import cmaps as gvcmaps  # Optional, check if available

# === Tropical Cyclone Analysis ===
from tropycal import tracks, utils

In [95]:
class ShockDecayAnalyzerPreloaded:
    def __init__(self, ref_period, shock_period, vertical_dim='plev', plot_dir=None):
        self.ref_period = ref_period
        self.shock_period = shock_period
        self.vertical_dim = vertical_dim
        self.plot_dir = plot_dir or "./"
        self.results = {}

        os.makedirs(self.plot_dir, exist_ok=True)

    def exp_decay(self, t, a0, tau, c):
        return a0 * np.exp(-t / tau) + c

    def _global_weighted_mean(self, da):
        weights = np.cos(np.deg2rad(da.lat))
        return da.weighted(weights).mean(dim=("lat", "lon"))

    def _compute_anomaly(self, da_mean, key='hour'): 
        shock = da_mean.sel(time=self.shock_period)
        if key == 'hour':
            clim = da_mean.sel(time=self.ref_period).groupby("time.hour").mean("time")
            anom = shock.groupby("time.hour") - clim
        elif key == 'dayofyear':
            clim = da_mean.sel(time=self.ref_period).groupby("time.dayofyear").mean("time")
            anom = shock.groupby("time.dayofyear") - clim
        else:
            raise ValueError(f"Unsupported anomaly grouping key: {key}")

        # Restore time axis for rolling and plotting
        return anom.reset_coords(drop=True).sortby("time")

    def _fit_decay(self, t, y):
        mask = ~np.isnan(y)
        if np.sum(mask) < 10 or np.all(np.isnan(y)):
            return np.nan, None
        try:
            a0_init = max(0, y[mask][0])  # ensure non-negative
            tau_init = 5
            c_init = 0
            popt, _ = curve_fit(
                self.exp_decay,
                t[mask],
                y[mask],
                p0=(a0_init, tau_init, c_init),
                bounds=([0, 0.1, -np.inf], [np.inf, 1e3, np.inf])
            )
            return popt[1], popt
        except RuntimeError:
            return np.nan, None

    def analyze_variable(self, varname, data, frequency='hourly', vertical_profile=False):
        self.results[varname] = {}
        nmem = data.sizes['member']

        for i in range(nmem):
            mem_str = f"EN{i:02d}"
            print(f"Analyzing {varname} - {mem_str}")
            da = data.isel(member=i)
            
            if vertical_profile and self.vertical_dim in da.dims:
                taus = {}
                for lev in da[self.vertical_dim].values:
                    try:
                        da_lev = da.sel({self.vertical_dim: lev})
                        mean = self._global_weighted_mean(da_lev)
                        anom = self._compute_anomaly(mean, frequency)
                        anom = anom.chunk({'time': -1})  # consolidate to a single chunk
                        window = min(5, max(3, anom.time.size // 10))
                        anom = anom.rolling(time=window, center=True).mean()
                        
                        t = np.arange(anom.time.size)
                        tau, popt = self._fit_decay(t, anom.values)
                        taus[float(lev)] = tau
                        self._plot_fit(t, anom.values, popt, varname, mem_str, lev)
                    except Exception as err:
                        print(f"  Level {lev}: {err}")
                        taus[float(lev)] = np.nan
                self.results[varname][mem_str] = taus
            else:
                try:
                    mean = self._global_weighted_mean(da)
                    anom = self._compute_anomaly(mean, frequency)
                    anom = anom.chunk({'time': -1})  # consolidate to a single chunk
                    window = min(5, max(3, anom.time.size // 10))
                    anom = anom.rolling(time=window, center=True).mean()                    
                    t = np.arange(anom.time.size)
                    tau, popt = self._fit_decay(t, anom.values)
                    self.results[varname][mem_str] = tau
                    self._plot_fit(t, anom.values, popt, varname, mem_str)
                except Exception as err:
                    print(f"  Error in {mem_str}: {err}")
                    self.results[varname][mem_str] = np.nan

    def _plot_fit(self, t, y, popt, varname, mem, lev=None):
        plt.figure(figsize=(8, 4))
        plt.plot(t, y, label="Anomaly", color="blue")
        if popt is not None:
            plt.plot(t, self.exp_decay(t, *popt), "r--", label=f"Fit (τ ≈ {popt[1]:.1f} d)")
        label = f"{varname}_{mem}" if lev is None else f"{varname}_{mem}_lev{int(lev)}"
        plt.title(f"Decay of {label}")
        plt.xlabel("Time index (e.g., days or steps)")
        plt.ylabel("Anomaly")
        plt.axhline(0, color="gray", linestyle=":")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(self.plot_dir, f"shock_decay_{label}.png"))
        plt.close()

    def summarize_results(self):
        flat_results = {}
        for var, vdict in self.results.items():
            for mem, val in vdict.items():
                if isinstance(val, dict):  # vertical profile
                    for lev, tau in val.items():
                        flat_results[(var, mem, lev)] = tau
                else:
                    flat_results[(var, mem, None)] = val
        df = pd.Series(flat_results).unstack([0, 1])
        return df

    def save_results(self, filename="shock_decay_timescales.csv"):
        df = self.summarize_results()
        df.to_csv(os.path.join(self.plot_dir, filename))


In [96]:
class ModelDataReader:
    """
    A reader class for loading and processing climate model output variables
    with standardized unit conversion and longitude correction. Supports ensemble reading.
    """

    def __init__(self, base_path, exp_dict):
        """
        Initialize the reader.

        Parameters
        ----------
        base_path : str
            Base directory for model outputs.
        exp_dict : dict
            Dictionary containing experiment metadata:
                - 'run': subdirectory name
                - 'clim': climatology folder name
                - 'period': time string in file names (e.g., '185001-186012')
        """
        self.base_path = base_path
        self.exp_dict = exp_dict
        
    def _convert_model_units(self, var, data):
        """
        Convert model output to standard units.

        Parameters
        ----------
        var : str
            Variable name (e.g., 'PRECT', 'TREFHT').
        data : xarray.DataArray
            Model data array.

        Returns
        -------
        xarray.DataArray
            Data array with standardized units.
        """
        if var == 'PRECT':
            return data * 86400.0 * 1000.0  # kg/m²/s to mm/day
        elif var in ['TREFHT', 'TS', 'T850']:
            return data - 273.15  # K to °C
        elif var in ['PSL', 'PS']:
            return data / 100.0  # Pa to hPa
        return data

    def read_variable(self, var, exp, frequency=None):
        """
        Load and process a model variable for a given experiment, optionally combining ensemble members.

        Parameters
        ----------
        var : str
            Variable name.
        exp : str
            Experiment key in exp_dict.
        frequency : str, optional
            Time frequency (currently unused).
        nens : int
            Number of ensemble members.

        Returns
        -------
        xarray.DataArray
            Combined model data with ensemble dimension (if nens > 1), units converted.
        """
        period = self.exp_dict[exp]['period'].replace("-", "")
        run = self.exp_dict[exp]['run']
        prefix = self.exp_dict[exp]['clim']
        nens = self.exp_dict[exp]['nens']
        ensemble_data = []

        for i in np.arange(1,nens+1,1):
            member = f"EN{i:02d}"
            search_dir = os.path.join(self.base_path, run, prefix)
            file_pattern = f"{var}.{member}.{period}.nc"
            search_path = os.path.join(search_dir, file_pattern)

            rpath = sorted(glob.glob(search_path))
            if not rpath:
                raise FileNotFoundError(f"No files found for {member} with pattern: {search_path}")

            ds = xcd.open_mfdataset(
                rpath,
                combine='nested',
                concat_dim='time',
                coords='minimal',
                compat='override'
            )

            if ds.lon.min() >= 0:
                ds = ds.assign_coords(lon=((ds.lon + 180) % 360 - 180)).sortby("lon")

            if var not in ds:
                raise KeyError(f"Variable '{var}' not found in dataset: {rpath[0]}")

            data = self._convert_model_units(var, ds[var])
            ensemble_data.append(data.expand_dims(member=[i]))

        combined = xr.concat(ensemble_data, dim='member')
        return combined.chunk({'member': -1, 'time': -1, 'lat': -1, 'lon': -1})

In [97]:
class ObsDataReader:
    """
    Class to read and preprocess observational data with internal dataset registry.
    """
    def __init__(self):
        self.obs_group = self._define_obs_group()

    def _define_obs_group(self):
        """
        Define observational datasets and their associated metadata.

        Returns
        -------
        dict
        """
        return {
            'ERA5': {
                'path': '/compyfs/zhan391/acme_init/Observations/ERA5/monthly',
                'file': '2011.nc',
                'vars': ['TS','TREFHT','PRECT', 'PSL', 'T850', 'Q850', 'U850', 'V850', 'Z500'],
            },
            'GPCP': {
                'path': '/compyfs/zhan391/acme_init/Observations/GPCP/monthly',
                'file': '2009-2011.nc',
                'vars': ['PRECT'],
            }
        }

    def get_group(self, name=None):
        if name:
            return self.obs_group.get(name)
        return self.obs_group

    def list_datasets(self):
        return list(self.obs_group.keys())

    def read(self, var, obsname, period, frequency=None):
        """
        Read and process observational data for a specific variable, dataset, and period.

        Parameters
        ----------
        var : str
            Variable name (e.g., 'PRECT', 'T850').
        obsname : str
            Dataset name (e.g., 'ERA5').
        period : str
            Target period in 'YYYY-MM' format.
        frequency : str or None
            Frequency of data (currently unused).

        Returns
        -------
        xarray.DataArray
        """
        obs_group = self.get_group(obsname)
        if obs_group is None:
            raise ValueError(f"Observational dataset '{obsname}' not found.")

        if var not in obs_group['vars']:
            raise ValueError(f"Variable '{var}' not available in '{obsname}' dataset.")

        obs_path = os.path.join(obs_group["path"], obs_group["file"])

        try:
            year, month = period.split("-")
        except ValueError:
            raise ValueError(f"Invalid period format '{period}'. Expected 'YYYY-MM'.")

        refds = xr.open_mfdataset(obs_path, decode_times=True, combine='by_coords')
        refds = refds.sel(time=f"{year}-{month}", drop=True)

        if refds.lon.min() >= 0:
            refds = refds.assign_coords(lon=((refds.lon + 180) % 360 - 180))
            refds = refds.sortby("lon")

        if var not in refds:
            raise KeyError(f"Variable '{var}' not found in file: {obs_path}")

        data = self.convert_obs_units(var, refds[var])
        data = data.chunk({"lat": -1, "lon": -1})

        return data

    def convert_obs_units(self, var, data_array):
        """
        Convert units and average over time.

        Parameters
        ----------
        var : str
            Variable name.
        data_array : xarray.DataArray

        Returns
        -------
        xarray.DataArray
        """
        if 'time' not in data_array.dims:
            raise ValueError(f"Expected 'time' dimension in data for '{var}'")

        data_mean = data_array.mean(dim='time')

        conversions = {
            'PRECT': lambda x: x * 86400.0 * 1000.0,
            'TREFHT': lambda x: x - 273.15,
            'TS': lambda x: x - 273.15,
            'T850': lambda x: x - 273.15,
            'PSL': lambda x: x / 100.0,
            'PS': lambda x: x / 100.0,
        }

        if var in conversions:
            return conversions[var](data_mean)
        else:
            print(f"[INFO] No unit conversion applied for variable '{var}'")
            return data_mean

In [98]:
def extract_exp_info(base_name: str = "", 
                     clim_path: str = "", 
                     period: str = "") -> dict:
    """
    Construct a dictionary of experiment metadata.

    Parameters
    ----------
    base_name : str, optional
        Common base name used in experiment run directory naming.
    clim_path : str, optional
        Path to the climatology output relative to the run directory.
    period : str, optional
        Time period used in file matching (e.g., 'YYYYMM').

    Returns
    -------
    dict
        Dictionary mapping experiment names to metadata dictionaries with:
        'run'   : constructed run directory name,
        'key'   : short alias for experiment group,
        'clim'  : climatology path,
        'period': time slice string,
        'nens'  : number of ensemble members.
    """
    exps = {
        'CTRLEN10':       {'name': 'ctrl_en10', 'nens': 10},
        'CAPTEN10_15day': {'name': 'capt_en10', 'nens': 10},
        'DARTEN10':       {'name': 'dart_en10', 'nens': 10},
        'DARTEN20':       {'name': 'dart_en20', 'nens': 20},
        'DARTEN40':       {'name': 'dart_en40', 'nens': 40},
        'DARTEN80':       {'name': 'dart_en80', 'nens': 80},
    }

    exp_dict = {
        exp_name: {
            'run': f"{exp_name}_{base_name}",
            'key': exp_data['name'],
            'nens': exp_data['nens'],
            'clim': clim_path,
            'period': period
        }
        for exp_name, exp_data in exps.items()
    }

    return exp_dict

In [101]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f'{compset}_{resolution}_{machine}'

    period = "2012-01"
    frequency = "monthly"

    var_dict = {
        'TS':      {'unit': '$^o$C',         'level': np.linspace(-8, 8, 51), 'ref': "ERA5"},
        'TREFHT':  {'unit': '$^o$C',         'level': np.linspace(-8, 8, 51), 'ref': "ERA5"},
        'PRECT':   {'unit': 'mm day$^{-1}$', 'level': np.linspace(-2, 2, 51), 'ref': "ERA5"},
        'PSL':     {'unit': 'hPa',           'level': np.linspace(-10, 10, 51), 'ref': "ERA5"},
    }

    model_list = ['CAPTEN10_15day']

    exp_dict = extract_exp_info(
        base_name=exp_base,
        clim_path='archive/post/atm/180x360_aave/ts/6hourly',
        period=period
    )
    model_reader = ModelDataReader(top_path, exp_dict)

    analyzer = ShockDecayAnalyzerPreloaded(
        ref_period=slice("2012-01-01", "2012-01-31"),
        shock_period=slice("2012-01-01", "2012-01-31"),
        plot_dir=os.path.join(fig_path, "initial_shock")
    )

    for model in model_list:
        for var in var_dict:
            mod = model_reader.read_variable(var, model, frequency=frequency)

            print(f"\n>>> Variable: {var} | Model: {model}")
            print(f"    Shape: {mod.shape} | Dims: {mod.dims}")

            if not isinstance(mod, xr.DataArray):
                raise TypeError(f"Expected DataArray for variable '{var}', got {type(mod)}")

            expected_dims = {'member', 'time', 'lat', 'lon'}
            if not expected_dims.issubset(mod.dims):
                raise ValueError(f"Unexpected dims in {var}: {mod.dims}, expected at least {expected_dims}")

            analyzer.analyze_variable(var, mod, frequency = 'hour', vertical_profile=False)
            
    summary = analyzer.summarize_results()
    print("\n=== Decay time scale summary (days) ===")
    print(summary.round(2))


>>> Variable: TS | Model: CAPTEN10_15day
    Shape: (10, 124, 180, 360) | Dims: ('member', 'time', 'lat', 'lon')
Analyzing TS - EN00
Analyzing TS - EN01
Analyzing TS - EN02
Analyzing TS - EN03
Analyzing TS - EN04
Analyzing TS - EN05
Analyzing TS - EN06
Analyzing TS - EN07
Analyzing TS - EN08
Analyzing TS - EN09

>>> Variable: TREFHT | Model: CAPTEN10_15day
    Shape: (10, 124, 180, 360) | Dims: ('member', 'time', 'lat', 'lon')
Analyzing TREFHT - EN00
Analyzing TREFHT - EN01
Analyzing TREFHT - EN02
Analyzing TREFHT - EN03
Analyzing TREFHT - EN04
Analyzing TREFHT - EN05
Analyzing TREFHT - EN06
Analyzing TREFHT - EN07
Analyzing TREFHT - EN08
Analyzing TREFHT - EN09

>>> Variable: PRECT | Model: CAPTEN10_15day
    Shape: (10, 124, 180, 360) | Dims: ('member', 'time', 'lat', 'lon')
Analyzing PRECT - EN00
Analyzing PRECT - EN01
Analyzing PRECT - EN02
Analyzing PRECT - EN03
Analyzing PRECT - EN04
Analyzing PRECT - EN05
Analyzing PRECT - EN06
Analyzing PRECT - EN07
Analyzing PRECT - EN08
Anal